# Assignment 5, Question 6: Data Transformation

**Points: 20**

Transform and engineer features from the clinical trial dataset.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities
from q3_data_utils import load_data, clean_data, transform_types, create_bins, fill_missing

df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df)} patients")

if "bmi" in df.columns:
        df.loc[df["bmi"] <= 0, "bmi"] = np.nan

if "age" in df.columns:
        df.loc[df["age"] <= 0, "age"] = np.nan

# Prewritten visualization functions for transformation analysis
def plot_distribution(series, title, figsize=(10, 6)):
    """
    Create a histogram of a numeric series.
    
    Args:
        series: pandas Series with numeric data
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.hist(bins=30)
    plt.title(title)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

def plot_value_counts(series, title, figsize=(10, 6)):
    """
    Create a bar chart of value counts.
    
    Args:
        series: pandas Series with value counts
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.plot(kind='bar')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Part 1: Type Conversions (5 points)

1. Convert 'enrollment_date' to datetime using the `transform_types()` utility
2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
3. Ensure all numeric columns are proper numeric types
4. Display the updated dtypes

In [ ]:
# TODO: Type conversions

df = transform_types(df, {
    'enrollment_date': 'datetime',
    'site': 'category',
    'intervention_group': 'category',
    'sex': 'category', })
# Verify type conversions
print("Data types after transformation:")
print(df.dtypes)



## Part 2: Feature Engineering (8 points)

Create these new calculated columns:

1. `cholesterol_ratio` = cholesterol_ldl / cholesterol_hdl
2. `bp_category` = categorize systolic BP:
   - 'Normal': < 120
   - 'Elevated': 120-129
   - 'High': >= 130
3. `age_group` using `create_bins()` utility:
   - Bins: [0, 40, 55, 70, 100]
   - Labels: ['<40', '40-54', '55-69', '70+']
4. `bmi_category` using standard BMI categories:
   - Underweight: <18.5
   - Normal: 18.5-24.9
   - Overweight: 25-29.9
   - Obese: >=30

In [ ]:
# TODO: Calculate cholesterol ratio
df['cholesterol_ratio'] = (df['cholesterol_ldl'] / df['cholesterol_hdl'])
print(df[['cholesterol_ldl', 'cholesterol_hdl', 'cholesterol_ratio']].head())




In [ ]:
# TODO: Categorize blood pressure
df['bp_category'] = np.select(
    [
        df['systolic_bp'] < 120,
        (df['systolic_bp'] >= 120) & (df['systolic_bp'] < 130),
        (df['systolic_bp'] >= 130)
    ],
    ['Normal', 'Elevated', 'High'   
    ],
    default='Unknown'
)
print(df[['systolic_bp', 'bp_category']].head())

**Note:** The `create_bins()` function has an optional `new_column` parameter. If you don't specify it, the new column will be named `{original_column}_binned`. You can use `new_column='age_group'` to give it a custom name.


In [ ]:
# TODO: Create age groups
df = create_bins(
    df,
    column = "age",
    bins=[0, 40, 55, 70, 100],
    labels=['<40', '40-54', '55-69', '70+'],
    new_column='age_group'
)
print(df[['age', 'age_group']].head())

In [ ]:
# TODO: Create BMI categories
df['bmi_category'] = np.select(
    [
        df['bmi'] < 18.5,
        (df['bmi'] >= 18.5) & (df['bmi'] < 24.9),
        (df['bmi'] >= 25) & (df['bmi'] < 29.9),
        df['bmi'] >= 30
    ],
    ['Underweight', 'Normal weight', 'Overweight', 'Obese'],
    default='Unknown'
)
print(df[['bmi', 'bmi_category']].head())

## Part 3: String Cleaning (2 points)

If there are any string columns that need cleaning:
1. Convert to lowercase
2. Strip whitespace
3. Replace any placeholder values

In [ ]:
# TODO: String cleaning
# Clean and standardize site names and intervention groups
for col in ["site", "intervention_group"]:
        df[col] = (
         df[col]
            .str.strip()
            .str.replace("_", " ")
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
    )

# Apply custom fixes for intervention_group
df["intervention_group"] = (
    df["intervention_group"]
    .str.replace("Treatmenta", "Treatment A")
    .str.replace("Treatmen A", "Treatment A")
    .str.replace("Contrl", "Control")
)

df["sex"] = (
    df["sex"].astype(str)
              .str.strip()
              .str[0]            
              .str.upper()
              .map({"M": "Male", "F": "Female"})
)



print(df[['site', 'intervention_group', 'sex']].head())

## Part 4: One-Hot Encoding (5 points)

Create dummy variables for categorical columns:
1. One-hot encode 'intervention_group' using `pd.get_dummies()`
2. One-hot encode 'site'
3. Drop the original categorical columns
4. Show the new shape and column names

In [ ]:
# TODO: One-hot encoding

cols = ["intervention_group", "site"]
encoded = pd.get_dummies(df, columns=cols, prefix=cols, drop_first=False)
print(df.filter(like='site_').head())


print("New Shape:", encoded.shape)
print("Columns:", encoded.columns.tolist())



## Part 5: Save Transformed Data

Save the fully transformed dataset to `output/q6_transformed_data.csv`

In [ ]:
# TODO: Save transformed data
df_transformed_encoded = encoded
df_transformed = df
df_transformed_encoded.to_csv('output/q6_transformed_data_encoded.csv', index=False)
df_transformed.to_csv('output/q6_transformed_data.csv', index=False)
